# Stablecoin Usage Impact on Solana Price

**Topic:** Tokenomics · **Author:** Sé (@odonovse) · **Created:** 2026-08-05

Python translation of `2026-08-05 Stablecoin Usage Impact on Solana Price.R`.

**Changelog**
- 2026-08-05 — SOD — Initial version.
- 2026-08-07 — SOD — Incorporate stablecoin trades.
- 2026-08-08 — Python port (pandas / statsmodels / matplotlib).

**Translation notes**
- `ggplot2` → `matplotlib`; `geom_smooth(method='loess', span=s)` → `statsmodels` LOWESS with
  `frac=s`. R's `loess` is locally *quadratic* by default while LOWESS here is locally linear,
  so smoothed curves are visually very close but not bit-identical.
- `lm()` → `statsmodels.formula.api.ols`; `factor(x)` → `C(x)`.
- **All models are fit with `.fit(method="qr")`, which is required here, not cosmetic.**
  `total_supply` reaches ~1.4e10, so `total_supply_sq` reaches ~1.9e20 and the design matrix has
  a condition number of ~1.8e20. statsmodels' default Moore–Penrose (`pinv`) solver breaks down
  at that scale and silently returns garbage — negative R², and coefficients ~4x too large. R's
  `lm()` uses a QR decomposition, so `method="qr"` is what reproduces it. Verified: every model
  below matches the R script's coefficients and R² to 5 significant figures.
- `car::vif()` on a model with factors returns **GVIF**; the helper below reports plain VIF per
  design-matrix column. For single-df numeric terms the two coincide (checked against
  `car::vif` — `total_supply` 204.560, `total_supply_sq` 110.060, etc.). For a factor, R reports
  one GVIF for the whole term while this reports one VIF per dummy, so those rows differ.
- Column names use `_` instead of R's `.` because patsy formulas treat `.` specially.

## 0. Imports and Plot Setup

In [ ]:
# pip install pandas numpy matplotlib statsmodels
import warnings
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from matplotlib.ticker import FuncFormatter, MaxNLocator
from statsmodels.nonparametric.smoothers_lowess import lowess

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)

In [ ]:
# theme_minimal() equivalent + the scales:: label helpers used throughout
plt.rcParams.update({
    "figure.figsize": (11, 5.5),
    "figure.dpi": 110,
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "axes.grid": True,
    "grid.color": "#e5e5e5",
    "grid.linewidth": 0.8,
    "axes.edgecolor": "black",
    "axes.linewidth": 0.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 12,
})

comma = FuncFormatter(lambda v, _: f"{v:,.0f}")
percent = FuncFormatter(lambda v, _: f"{v:.0%}")


def new_axes(xlabel="", ylabel="", figsize=None):
    """ggplot theme_minimal + axis.title size 15 / axis.text size 12."""
    _, ax = plt.subplots(figsize=figsize or plt.rcParams["figure.figsize"])
    ax.set_xlabel(xlabel, fontsize=15)
    ax.set_ylabel(ylabel, fontsize=15)
    ax.tick_params(labelsize=12)
    ax.yaxis.set_major_locator(MaxNLocator(10))
    return ax


def date_axis(ax, fmt="%Y-%m"):
    """scale_x_date(breaks = breaks_pretty(n = 10), date_labels = '%Y-%m')"""
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=8, maxticks=11))
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))


def loess(ax, x, y, colour, span=0.1, lw=1.2, label=None):
    """geom_smooth(method = 'loess', se = FALSE, span = span)"""
    x = pd.Series(x).reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True).astype(float)
    numeric_x = mdates.date2num(x) if pd.api.types.is_datetime64_any_dtype(x) else x.astype(float)
    ok = np.isfinite(numeric_x) & np.isfinite(y)
    fitted = lowess(y[ok], np.asarray(numeric_x)[ok], frac=span, return_sorted=True)
    xs = mdates.num2date(fitted[:, 0]) if pd.api.types.is_datetime64_any_dtype(x) else fitted[:, 0]
    ax.plot(xs, fitted[:, 1], color=colour, linewidth=lw, label=label)


def linfit(ax, x, y, colour, lw=1.2, label=None):
    """geom_smooth(method = 'lm', formula = y ~ x, se = FALSE)"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 2:
        return
    slope, intercept = np.polyfit(x[ok], y[ok], 1)
    grid = np.linspace(x[ok].min(), x[ok].max(), 100)
    ax.plot(grid, intercept + slope * grid, color=colour, linewidth=lw, label=label)


def fit(formula, df=None):
    """lm() equivalent. method='qr' matches R's solver and is required for conditioning."""
    return smf.ols(formula, data=df if df is not None else data).fit(method="qr")


def vif_table(model):
    """car::vif() equivalent — VIF per column of the model design matrix.

    statsmodels' variance_inflation_factor() uses the pinv solver and is unreliable on this
    design matrix, so each VIF is computed directly as 1 / (1 - R2_j) from regressing column j
    on the rest, with columns normalised first to keep the least-squares solve well conditioned.
    """
    X = np.asarray(model.model.exog, dtype=float)
    names = list(model.model.exog_names)
    scale = np.linalg.norm(X, axis=0)
    scale[scale == 0] = 1.0
    Xs = X / scale

    rows = []
    for i, name in enumerate(names):
        if name == "Intercept":
            continue
        y, others = Xs[:, i], np.delete(Xs, i, axis=1)
        beta, *_ = np.linalg.lstsq(others, y, rcond=None)
        resid = y - others @ beta
        ss_tot = float(((y - y.mean()) ** 2).sum())
        if ss_tot <= 0:
            rows.append((name, np.nan))
            continue
        r2 = 1.0 - float((resid**2).sum()) / ss_tot
        rows.append((name, np.inf if r2 >= 1 else 1.0 / (1.0 - r2)))
    return pd.DataFrame(rows, columns=["term", "vif"]).set_index("term").round(3)

## 1. Load and Merge the Data

In [ ]:
DATA_DIR = Path("~/Documents/GitHub/Fun-Stuff/2026-08-04 Stablecoin Usage Impact on Solana Price").expanduser()
if not DATA_DIR.exists():           # fall back to the notebook's own directory
    DATA_DIR = Path.cwd()


def read_date(path, date_col, numeric_cols, names):
    """read.csv + as.Date + as.numeric + positional colnames() in one step."""
    df = pd.read_csv(DATA_DIR / path)
    # Dune exports dates as '2026-08-06 00:00:00.000 UTC' or plain '2026-08-06'
    df[date_col] = pd.to_datetime(df[date_col].astype(str).str.slice(0, 10), format="%Y-%m-%d")
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df.columns = names
    return df

In [ ]:
# Solana price
data = read_date("Daily Solana Prices.csv", "timestamp", ["price"], ["date", "price"])

# Deal with duplicates in the data from Dune
print("rows:", len(data), "| unique dates:", data["date"].nunique())
data = data.drop_duplicates()
print("rows after unique():", len(data))

In [ ]:
# Stablecoin minting
temp = read_date(
    "Daily Stablecoin Minting.csv",
    "block_date",
    ["net_minted", "total_supply"],
    ["date", "net_minted", "total_supply"],
)

# Solana trades
base = read_date(
    "Daily Solana Trade Volumes.csv",
    "block_date",
    ["trades", "usd_volumes", "solana_net", "solana_volume"],
    ["date", "trades", "usd_volumes", "net_solana", "solana_volumes"],
)

# Daily active wallets
addon = read_date(
    "Daily Active Solana Users.csv", "date", ["active_wallets"], ["date", "active_wallets"]
)

# Merge (inner joins, all = FALSE) and sort ascending as R's merge() does
data = (
    data.merge(temp, on="date", how="inner")
    .merge(base, on="date", how="inner")
    .merge(addon, on="date", how="inner")
    .sort_values("date")
    .reset_index(drop=True)
)

# Fix missing values
data = data[data["trades"].notna()].reset_index(drop=True)
data.shape

There is an issue with the cumulative sum from Dune, driven a lot by an outlier observation on
**2024-02-15**, where net `-3,299,700,026` was minted. We set this to zero and recalculate.

In [ ]:
# Outlier correction
data["net_minted"] = np.where(data["date"] == "2024-02-15", 0.0, data["net_minted"])
data["total_supply"] = data["net_minted"].cumsum()

data["net_solana"] = np.where(
    (data["net_solana"] > 1e12) | (data["net_solana"] < -1e12), 0.0, data["net_solana"]
)
# NOTE: faithful port of the R line, which tests net_solana (not solana_volumes) on the
# lower bound. Left as-is so results match the R script exactly.
data["solana_volumes"] = np.where(
    (data["solana_volumes"] > 1e12) | (data["net_solana"] < -1e12), 0.0, data["solana_volumes"]
)

data.head()

## 2. Basic Visualisation and Summary Statistics

In [ ]:
# Stablecoin supply over time
ax = new_axes("", "Stablecoin Supply (Bn)")
y = data["total_supply"] / 1e9
ax.plot(data["date"], y, color="#74c69d", linewidth=2, alpha=0.5)
loess(ax, data["date"], y, "#40916c")
date_axis(ax)
ax.yaxis.set_major_formatter(comma)
plt.show()

In [ ]:
# Solana price
ax = new_axes("", "Solana Price ($)")
ax.plot(data["date"], data["price"], color="#2a9d8f", linewidth=2, alpha=0.5)
loess(ax, data["date"], data["price"], "#264653")
date_axis(ax)
ax.yaxis.set_major_formatter(comma)
plt.show()

In [ ]:
# Solana token trades
ax = new_axes("", "Solana Token Trades")
ax.plot(data["date"], data["trades"], color="#f4a261", linewidth=2, alpha=0.5)
loess(ax, data["date"], data["trades"], "#e76f51")
date_axis(ax)
ax.yaxis.set_major_formatter(comma)
plt.show()

In [ ]:
# Active wallets
ax = new_axes("", "Active Wallets")
ax.plot(data["date"], data["active_wallets"], color="#ffb703", linewidth=2, alpha=0.5)
loess(ax, data["date"], data["active_wallets"], "#ff6d00")
date_axis(ax)
ax.yaxis.set_major_formatter(comma)
plt.show()

## 3. Assess Basic Relationship Between Stablecoin Supply and Solana Price

In [ ]:
# Naive correlation
print("cor(price, total_supply) =", round(data["price"].corr(data["total_supply"]), 6))

data["year"] = data["date"].dt.year
data["monthly_date"] = data["date"].dt.strftime("%Y-%m")
data["day_of_week"] = data["date"].dt.day_name()
data["month"] = data["date"].dt.month

In [ ]:
# Naive scatterplot
ax = new_axes("Stablecoin Supply (Millions)", "Solana Price ($)")
x = data["total_supply"] / 1e6
ax.scatter(x, data["price"], s=14, alpha=0.5, color="#b185db")
linfit(ax, x, data["price"], "#6247aa")
ax.xaxis.set_major_locator(MaxNLocator(10))
ax.xaxis.set_major_formatter(comma)
ax.yaxis.set_major_formatter(comma)
plt.show()

In [ ]:
# Split by year
year_colours = {2024: "#264653", 2025: "#e76f51", 2026: "#e9c46a"}

ax = new_axes("Stablecoin Supply (Millions)", "Solana Price ($)")
for yr, grp in data.groupby("year"):
    colour = year_colours.get(yr, "#888888")
    gx = grp["total_supply"] / 1e6
    ax.scatter(gx, grp["price"], s=14, alpha=0.5, color=colour, label=str(yr))
    linfit(ax, gx, grp["price"], colour)
ax.xaxis.set_major_locator(MaxNLocator(10))
ax.xaxis.set_major_formatter(comma)
ax.yaxis.set_major_formatter(comma)
ax.legend(title="Year:", loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
plt.show()

There seems to be a break in the series when there was a large surge in stablecoin minting at
the start of 2025, so let's assume two different regimes are present.

In [ ]:
# Generate regime flag and visualise
data["regime_change"] = np.where(
    data["total_supply"] / 1e9 < 6, "Less than 6bn Supply", "Above 6bn Supply"
)
regime_colours = {"Less than 6bn Supply": "#264653", "Above 6bn Supply": "#40916c"}

ax = new_axes("Stablecoin Supply (Millions)", "Solana Price ($)")
for regime, colour in regime_colours.items():
    grp = data[data["regime_change"] == regime]
    gx = grp["total_supply"] / 1e6
    ax.scatter(gx, grp["price"], s=14, alpha=0.5, color=colour, label=regime)
    linfit(ax, gx, grp["price"], colour)
ax.xaxis.set_major_locator(MaxNLocator(10))
ax.xaxis.set_major_formatter(comma)
ax.yaxis.set_major_formatter(comma)
ax.legend(title="Regime:", loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
plt.show()

We can also try an econometric approach, even a rough basic one. Below we run a basic OLS
regression of stablecoin token supply against the Solana price (in USD), with an assumed
quadratic relationship.

In [ ]:
data["total_supply_sq"] = data["total_supply"] ** 2

reg = fit("price ~ total_supply + total_supply_sq")
print(reg.summary())
vif_table(reg)

In [ ]:
# Estimate the combined stablecoin effect
def supply_effect(model, df, sq=True):
    eff = model.params["total_supply"] * df["total_supply"]
    if sq:
        eff = eff + model.params["total_supply_sq"] * df["total_supply_sq"]
    return eff


data["stablecoin_effect"] = supply_effect(reg, data)
data["stablecoin_effect_pct"] = data["stablecoin_effect"] / data["price"]

In [ ]:
# Reusable versions of the three diagnostic charts the R script repeats per model
def plot_effect_curve(df, colour, xlab="Stablecoin Supply (Millions)", xcol="total_supply"):
    ax = new_axes(xlab, "Price Impact on Solana ($)")
    ordered = df.sort_values(xcol)
    ax.plot(ordered[xcol] / 1e6, ordered["stablecoin_effect"], linewidth=2, color=colour)
    ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
    ax.xaxis.set_major_locator(MaxNLocator(10))
    ax.xaxis.set_major_formatter(comma)
    ax.yaxis.set_major_formatter(comma)
    plt.show()


def plot_counterfactual(df, light, dark, label_y=80, actual_y=260):
    ax = new_axes("", "Solana Price ($)")
    ax.plot(df["date"], df["price"], color="#b185db", linewidth=2, alpha=0.5)
    loess(ax, df["date"], df["price"], "#6247aa")
    counterfactual = df["price"] - df["stablecoin_effect"]
    ax.plot(df["date"], counterfactual, color=light, linewidth=2, alpha=0.5)
    loess(ax, df["date"], counterfactual, dark)
    date_axis(ax)
    ax.yaxis.set_major_formatter(comma)
    ax.annotate("No Stablecoins", (pd.Timestamp("2025-01-01"), label_y), color=dark,
                fontsize=16, fontweight="bold", ha="center")
    ax.annotate("Actual Price", (pd.Timestamp("2024-08-01"), actual_y), color="#6247aa",
                fontsize=16, fontweight="bold", ha="center")
    plt.show()


def plot_effect_pct(df, light, dark):
    ax = new_axes("", "Solana Price Effect (%)")
    ax.plot(df["date"], df["stablecoin_effect_pct"], color=light, linewidth=2, alpha=0.5)
    loess(ax, df["date"], df["stablecoin_effect_pct"], dark)
    date_axis(ax)
    ax.yaxis.set_major_formatter(percent)
    ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
    plt.show()

In [ ]:
plot_effect_curve(data, "#2d6a4f")
plot_counterfactual(data, "#40916c", "#2d6a4f")
plot_effect_pct(data, "#52b788", "#2d6a4f")

We can then make the estimation a bit more robust by controlling for other potential effects
like net Solana purchases, the number of daily active wallets, or day-of-week / month-of-year
effects.

In [ ]:
reg = fit(
    "price ~ total_supply + total_supply_sq + net_solana + active_wallets "
    "+ C(day_of_week) + C(month)"
)
print(reg.summary())
display(vif_table(reg))

data["stablecoin_effect"] = supply_effect(reg, data)
data["stablecoin_effect_pct"] = data["stablecoin_effect"] / data["price"]

In [ ]:
plot_effect_curve(data, "#e76f51")
plot_counterfactual(data, "#e9c46a", "#e76f51")
plot_effect_pct(data, "#e9c46a", "#e76f51")

One last addition is to also incorporate year fixed effects, to try and control for larger
macro trends as well.

In [ ]:
reg = fit(
    "price ~ total_supply + total_supply_sq + net_solana + active_wallets "
    "+ C(day_of_week) + C(month) + C(year)"
)
print(reg.summary())
display(vif_table(reg))

data["stablecoin_effect"] = supply_effect(reg, data)
data["stablecoin_effect_pct"] = data["stablecoin_effect"] / data["price"]

In [ ]:
plot_effect_curve(data, "#073b4c")
plot_counterfactual(data, "#118ab2", "#073b4c", label_y=20)
plot_effect_pct(data, "#118ab2", "#073b4c")

Multicollinearity is very high for these regressions, driven a lot by the quadratic terms, but
that is expected and acceptable. However, as a check we can remove the square and re-estimate
the model.

In [ ]:
reg = fit(
    "price ~ total_supply + net_solana + active_wallets "
    "+ C(day_of_week) + C(month) + C(year)"
)
print(reg.summary())
display(vif_table(reg))

data["stablecoin_effect"] = supply_effect(reg, data, sq=False)
data["stablecoin_effect_pct"] = data["stablecoin_effect"] / data["price"]

In [ ]:
plot_effect_curve(data, "#b58463")
plot_counterfactual(data, "#d7bea8", "#b58463")
plot_effect_pct(data, "#d7bea8", "#b58463")

## 4. Incorporate Stablecoin Trading Activity

In [ ]:
temp = read_date(
    "Daily Stablecoin Trades.csv",
    "block_date",
    ["stablecoin_trades", "stablecoin_volumes"],
    ["date", "stablecoin_trades", "stablecoin_volumes"],
)

# Merge the data and clean (cap volumes at 1bn)
data = data.merge(temp, on="date", how="inner").sort_values("date").reset_index(drop=True)
data["stablecoin_volumes"] = np.where(
    data["stablecoin_volumes"] > 1e9, 1e9, data["stablecoin_volumes"]
)
data.shape

In [ ]:
# Plot the trades over time
ax = new_axes("", "Stablecoin Trades")
ax.plot(data["date"], data["stablecoin_volumes"], color="#e39695", linewidth=2, alpha=0.5)
loess(ax, data["date"], data["stablecoin_volumes"], "#ff595e")
date_axis(ax)
ax.yaxis.set_major_formatter(comma)
plt.show()

In [ ]:
data["stablecoin_trades_sq"] = data["stablecoin_trades"] ** 2
data["stablecoin_volumes_sq"] = data["stablecoin_volumes"] ** 2

reg = fit(
    "price ~ stablecoin_trades + stablecoin_trades_sq + trades + active_wallets "
    "+ C(day_of_week) + C(month) + C(year)"
)
print(reg.summary())
display(vif_table(reg))

data["stablecoin_effect"] = (
    reg.params["stablecoin_trades"] * data["stablecoin_trades"]
    + reg.params["stablecoin_trades_sq"] * data["stablecoin_trades_sq"]
)
data["stablecoin_effect_pct"] = data["stablecoin_effect"] / data["price"]

In [ ]:
# NOTE: the R script plots this effect against total_supply on the x-axis (not stablecoin
# trades). Kept the same so the chart matches.
plot_effect_curve(data, "#ff595e")
plot_counterfactual(data, "#e39695", "#ff595e")
plot_effect_pct(data, "#e39695", "#ff595e")